In [1]:
# plot_level_correlation_yolo_v6.py
#
# Purpose: Compute correlation between field survey plant counts
# and model detection counts per plot
# Tests whether model detections reflect real plant abundance
# Uses predictions saved by model.val(save_txt=True)

import os
import re
import pandas as pd
import numpy as np
from scipy import stats

# ── model predictions (saved by save_txt=True) ────────────────────────────────
# predictions are saved per image as txt files in eval_summer/labels/
PRED_DIR_SUMMER = r"C:\Master Physische Geographie\Canada\Masterarbeit\results\field_plot_evaluation_yolo_v6\eval_summer\labels"
PRED_DIR_FALL   = r"C:\Master Physische Geographie\Canada\Masterarbeit\results\field_plot_evaluation_yolo_v6\eval_fall\labels"

# ── field survey counts per plot (from visibility analysis) ───────────────────
# n_field_3m = plants recorded within 3m radius
field_data = {
    0:  19,  1:  12,  2:  14,  3:   8,  4:  13,
    5:  12,  6:  11,  7:   1,  8:   0,  9:   4,
    10:  3,  11: 27,  12:  3,  13:  5,  14: 19,
    15: 16,  16: 12,  17:  8,  18:  0,  19: 12,
    20: 27,  21:  0,  22: 12,  23: 26,  24: 18,
    25: 13,  26: 46,  27:  2,  28:  4,  29: 13,
    30:  3,  31: 12,  32: 12,  33: 11,  34: 15,
    35: 10,  36: 15,  37: 13,  38: 27,  39:  5,
    40:  7,  41:  6,  42:  9,  43: 10,  44:  5,
    45:  9,  46:  8,  47:  9,  48:  4,  49:  0,
}

def count_predictions(pred_dir):
    """Count model detections per plot from saved prediction files"""
    counts = {}
    if not os.path.exists(pred_dir):
        print(f"WARNING: predictions folder not found: {pred_dir}")
        return counts

    for fname in os.listdir(pred_dir):
        if not fname.endswith(".txt"):
            continue

        # extract plot_id from filename
        match = re.search(r"_plot(\d+)_", fname)
        if not match:
            continue
        plot_id = int(match.group(1))

        with open(os.path.join(pred_dir, fname)) as f:
            lines = [l.strip() for l in f if l.strip()]

        n_dec = sum(1 for l in lines if l.startswith("0"))
        n_shr = sum(1 for l in lines if l.startswith("1"))
        counts[plot_id] = {
            "n_pred_deciduous": n_dec,
            "n_pred_shrub":     n_shr,
            "n_pred_total":     n_dec + n_shr,
        }
    return counts

# ── compute correlations ───────────────────────────────────────────────────────
def compute_correlation(pred_dir, season):
    print(f"\n{'='*50}")
    print(f"{season.upper()} — Plot-level correlation")
    print(f"{'='*50}")

    preds = count_predictions(pred_dir)

    rows = []
    for plot_id, n_field in field_data.items():
        pred = preds.get(plot_id, {"n_pred_total": 0,
                                   "n_pred_deciduous": 0,
                                   "n_pred_shrub": 0})
        rows.append({
            "plot_id":         plot_id,
            "n_field":         n_field,
            "n_pred_total":    pred["n_pred_total"],
            "n_pred_deciduous":pred["n_pred_deciduous"],
            "n_pred_shrub":    pred["n_pred_shrub"],
        })

    df = pd.DataFrame(rows)

    # exclude plots with 0 field plants (nothing to correlate)
    df_valid = df[df["n_field"] > 0].copy()
    print(f"Plots with field plants: {len(df_valid)}")
    print(f"Plots excluded (0 field plants): {len(df) - len(df_valid)}")

    # Pearson correlation
    r, p = stats.pearsonr(df_valid["n_field"], df_valid["n_pred_total"])
    print(f"\nPearson r (field vs predicted): {r:.3f}")
    print(f"p-value: {p:.4f}")
    print(f"Interpretation: {'significant (p<0.05)' if p < 0.05 else 'not significant'}")

    # Spearman correlation (rank-based, more robust)
    rho, p_rho = stats.spearmanr(df_valid["n_field"], df_valid["n_pred_total"])
    print(f"\nSpearman rho: {rho:.3f}")
    print(f"p-value: {p_rho:.4f}")

    # mean detection rate
    df_valid["detection_rate"] = df_valid["n_pred_total"] / df_valid["n_field"]
    print(f"\nMean detection rate: {df_valid['detection_rate'].mean():.1%}")
    print(f"Median detection rate: {df_valid['detection_rate'].median():.1%}")

    print(f"\nPer-plot breakdown:")
    print(df[["plot_id","n_field","n_pred_total",
              "n_pred_deciduous","n_pred_shrub"]].to_string(index=False))

    return df, r, p, rho, p_rho

summer_df, r_s, p_s, rho_s, p_rho_s = compute_correlation(PRED_DIR_SUMMER, "summer")
fall_df,   r_f, p_f, rho_f, p_rho_f = compute_correlation(PRED_DIR_FALL,   "fall")

# ── save ───────────────────────────────────────────────────────────────────────
output = r"C:\Master Physische Geographie\Canada\Masterarbeit\results\field_plot_evaluation_yolo_v6\plot_level_correlation_yolo_v6.csv"
combined = summer_df.rename(columns={
    "n_pred_total": "summer_pred_total",
    "n_pred_deciduous": "summer_pred_deciduous",
    "n_pred_shrub": "summer_pred_shrub"
}).merge(
    fall_df[["plot_id","n_pred_total","n_pred_deciduous","n_pred_shrub"]].rename(columns={
        "n_pred_total": "fall_pred_total",
        "n_pred_deciduous": "fall_pred_deciduous",
        "n_pred_shrub": "fall_pred_shrub"
    }), on="plot_id"
)
combined.to_csv(output, index=False)
print(f"\nSaved: {output}")

print(f"\n{'='*50}")
print("SUMMARY")
print(f"{'='*50}")
print(f"Summer Pearson r={r_s:.3f} (p={p_s:.4f})")
print(f"Summer Spearman rho={rho_s:.3f} (p={p_rho_s:.4f})")
print(f"Fall Pearson r={r_f:.3f} (p={p_f:.4f})")
print(f"Fall Spearman rho={rho_f:.3f} (p={p_rho_f:.4f})")


SUMMER — Plot-level correlation
Plots with field plants: 46
Plots excluded (0 field plants): 4

Pearson r (field vs predicted): 0.607
p-value: 0.0000
Interpretation: significant (p<0.05)

Spearman rho: 0.596
p-value: 0.0000

Mean detection rate: 1073.8%
Median detection rate: 989.5%

Per-plot breakdown:
 plot_id  n_field  n_pred_total  n_pred_deciduous  n_pred_shrub
       0       19           300               297             3
       1       12            49                47             2
       2       14           300                 5           295
       3        8            95                 2            93
       4       13             8                 8             0
       5       12           300                 0           300
       6       11           159                26           133
       7        1            12                10             2
       8        0             6                 4             2
       9        4            27                27     